# NFL Kaggle dataset

![game day rio](https://s2-ge.glbimg.com/U1mSZw9GaaiGTDxCiSLvZ8O15ho=/0x0:1031x1136/984x0/smart/filters:strip_icc()/i.s3.glbimg.com/v1/AUTH_bc8228b6673f488aa253bbcb03c80ec5/internal_photos/bs/2026/t/u/l1gitBSHuoPF44ya5Hzg/no-templo-do-futebol-um-confronto-historico.-voce-esta-pronto-in-the-temple-of-soccer-a.jpg)

### A primeira etapa é importar as bibliotecas que iremos utilizar 

In [27]:
import os
import numpy as np
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

In [28]:
# baixa o dataset do KaggleHub e retorna o caminho local onde os arquivos foram armazenados
root = kagglehub.dataset_download("maxhorowitz/nflplaybyplay2009to2016") 
print("Root:", root)

Root: C:\Users\47447309818\.cache\kagglehub\datasets\maxhorowitz\nflplaybyplay2009to2016\versions\6


In [29]:
for dirpath, dirnames, filenames in os.walk(root):  # percorre recursivamente todas as pastas dentro do diretório root
    for f in filenames:  # itera sobre cada arquivo encontrado em cada pasta
        print(os.path.relpath(os.path.join(dirpath, f), root))  # imprime o caminho relativo do arquivo em relação ao diretório root

NFL Play by Play 2009-2016 (v3).csv
NFL Play by Play 2009-2017 (v4) PROCESSED.parquet
NFL Play by Play 2009-2017 (v4).csv
NFL Play by Play 2009-2018 (v5).csv


In [30]:
# Exemplo: ajuste para o arquivo real que você viu no os.walk
csv_path = os.path.join(root, "NFL Play by Play 2009-2017 (v4).csv") # cria o caminho completo para o arquivo CSV dentro do diretório root

df = pd.read_csv(csv_path) # lê o arquivo CSV e carrega os dados em um DataFrame do pandas
df.head()

C:\Users\47447309818\AppData\Local\Temp\ipykernel_3148\848581222.py:4: DtypeWarning: Columns (25: DefTwoPoint, 51: BlockingPlayer) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path) # lê o arquivo CSV e carrega os dados em um DataFrame do pandas


,Date,GameID,Drive,qtr,down,time,TimeUnder,TimeSecs,PlayTimeDiff,SideofField,...,yacEPA,Home_WP_pre,Away_WP_pre,Home_WP_post,Away_WP_post,Win_Prob,WPA,airWPA,yacWPA,Season
0,2009-09-10,2009091000,1,1,NaN,15:00,15,3600.0,0.0,TEN,...,NaN,0.485675,0.514325,0.546433,0.453567,0.485675,0.060758,NaN,NaN,2009
1,2009-09-10,2009091000,1,1,1.0,14:53,15,3593.0,7.0,PIT,...,1.146076,0.546433,0.453567,0.551088,0.448912,0.546433,0.004655,-0.032244,0.036899,2009
2,2009-09-10,2009091000,1,1,2.0,14:16,15,3556.0,37.0,PIT,...,NaN,0.551088,0.448912,0.510793,0.489207,0.551088,-0.040295,NaN,NaN,2009
3,2009-09-10,2009091000,1,1,3.0,13:35,14,3515.0,41.0,PIT,...,-5.031425,0.510793,0.489207,0.461217,0.538783,0.510793,-0.049576,0.106663,-0.156239,2009
4,2009-09-10,2009091000,1,1,4.0,13:27,14,3507.0,8.0,PIT,...,NaN,0.461217,0.538783,0.558929,0.441071,0.461217,0.097712,NaN,NaN,2009


In [31]:
df.info(verbose=True) 

<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype  
---   ------                       -----  
 0    Date                         str    
 1    GameID                       int64  
 2    Drive                        int64  
 3    qtr                          int64  
 4    down                         float64
 5    time                         str    
 6    TimeUnder                    int64  
 7    TimeSecs                     float64
 8    PlayTimeDiff                 float64
 9    SideofField                  str    
 10   yrdln                        float64
 11   yrdline100                   float64
 12   ydstogo                      int64  
 13   ydsnet                       int64  
 14   GoalToGo                     float64
 15   FirstDown                    float64
 16   posteam                      str    
 17   DefensiveTeam                str    
 18   desc                         str 

### O que o DtypeWarning está dizendo? 
`Columns (25,51) have mixed types.`

Significa que, nas colunas de índice 25 e 51:

- Parte dos valores parece numérica

- Parte dos valores parece texto

- O pandas não conseguiu inferir um único dtype de forma segura

Por padrão, o `read_csv` lê o arquivo em chunks para economizar memória (`low_memory=True`), o que aumenta a chance desse aviso.

Alternativamente, pode-se ler o conjunto de dados como:
`df = pd.read_csv(csv_path, low_memory=False)`

Isso força leitura em uma única passada e reduz falsos positivos de inferência.

---
### Importante:
Em uma etapa de Análise Exploratória de Dados, sempre investique os alertas do tipo `DtypeWarning`
- Neste caso, vamos identificar e inspecionar essas colunas:


In [32]:
problem_cols = df.columns[[25, 51]]
problem_cols

Index(['DefTwoPoint', 'BlockingPlayer'], dtype='str')

### Inspecionado as informações

aplicar a função type em cada valor da coluna problem_cols[0] e conta quantas ocorrências existem de cada tipo de dado

In [33]:
df[problem_cols[0]].apply(type).value_counts()

DefTwoPoint
<class 'float'>    407664
<class 'str'>          24
Name: count, dtype: int64

In [34]:
df[problem_cols[1]].apply(type).value_counts()

BlockingPlayer
<class 'float'>    407571
<class 'str'>         117
Name: count, dtype: int64

- Inspecionando dados do tipo `str`

In [35]:
df.loc[df[problem_cols[0]].apply(type) == str, problem_cols[0]]

278907    Failure
284977    Failure
286715    Failure
301227    Failure
304879    Success
321075    Success
323901    Failure
341248    Success
342856    Failure
345982    Failure
348856    Success
351968    Success
352569    Failure
359061    Failure
362769    Failure
368022    Failure
372456    Failure
381748    Failure
385310    Failure
386506    Failure
390624    Failure
403182    Failure
404890    Failure
406623    Failure
Name: DefTwoPoint, dtype: str

- Inspecionando dados do tipo `float`

In [36]:
df.loc[df[problem_cols[0]].apply(type) == float, problem_cols[0]].head()

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: DefTwoPoint, dtype: str

In [37]:
df.loc[df[problem_cols[1]].apply(type) == float, problem_cols[1]].head()

0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
Name: BlockingPlayer, dtype: str

- Inpecionando dados únicos

In [38]:
df[problem_cols[0]].unique()

<ArrowStringArray>
[nan, 'Failure', 'Success']
Length: 3, dtype: str

In [39]:
df[problem_cols[1]].unique()

<ArrowStringArray>
[             nan,       'J.McGraw',     'B.Williams',       'J.Hester',
        'G.Hayes',        'E.Smith',        'C.Lynch',        'G.Hardy',
     'R.Jennings',        'D.Curry',
 ...
     'C.Reynolds',   'T.Matakevich',    'C.Littleton',       'J.Willis',
       'J.Hardee',     'R.Burkhead',      'S.Barrett',       'M.Thomas',
 'K.Grugier-Hill',       'A.Walker']
Length: 102, dtype: str

- Inspecionado valores nulos 

In [40]:
for cols in df[problem_cols]:
    print("Total de NaN em",cols,":",df[cols].isna().sum())

Total de NaN em DefTwoPoint : 407664
Total de NaN em BlockingPlayer : 407571


----------------------
---------------------

### Tratamento de Dtypewarning

Nesta etapa vamos pegar as duas variaveis  que estão apresentando o alerta de DtypeWarning que são as variaveis DefTwoPoint e BlockingPlayer e padronizar cada variavel para um unico tipo usando o astype()

- DefTwoPoint

    - Nesta etapa contamos quantos valores unicos possui a variavel que esta descrita como str e apartir da informação vimos que possui 3 valores contando com os valores ausentes portanto é uma variavel categorica e realizamos a conversão

In [41]:
df[problem_cols[0]].unique()

<ArrowStringArray>
[nan, 'Failure', 'Success']
Length: 3, dtype: str

In [42]:
df[problem_cols[0]] = df[problem_cols[0]].astype("category")  
df[problem_cols[0]].value_counts(dropna=False)

DefTwoPoint
NaN        407664
Failure        19
Success         5
Name: count, dtype: int64

- BlockingPlayer
    - Nesta etapa convertemos os valores da variavel para string pois ela trata de jogadores que realizam um tipo especifico de cada jogada incluindo os valores nulos por isso converti pra string para facilitar o seu tratamento
    - Nesta etapa também é possivel realizar o tratamento dos valores nulos onde cada time possui um aproveitamento neste tipo de jogada , com os jogadores de cada time que mais realizam cada tipo de jogada e apartir desta informação é possivel dar um valor para esses valores nulos

In [43]:
df[problem_cols[1]] = df[problem_cols[1]].astype("string")  

In [44]:
df[problem_cols[1]].value_counts(dropna=False)  

BlockingPlayer
<NA>              407571
R.Jennings             3
B.Braman               3
D.Watson               3
E.Smith                2
                   ...  
R.Burkhead             1
S.Barrett              1
M.Thomas               1
K.Grugier-Hill         1
A.Walker               1
Name: count, Length: 102, dtype: int64[pyarrow]

In [45]:
# 1. Filtrar apenas as jogadas onde houve registro de BlockingPlayer (remover nulos / <NA>)
df_blocking = df[df['BlockingPlayer'].notna() & (df['BlockingPlayer'] != '<NA>')].copy()

# 2. Agrupar por Time Defensivo e Jogador, contando o total de bloqueios efetuados
blocking_counts = (
    df_blocking.groupby(['DefensiveTeam', 'BlockingPlayer'])
    .size()
    .reset_index(name='Total_Bloqueios')
)

# 3. Identificar o jogador com o MAIOR número de bloqueios em cada time
top_blocker_por_time = (
    blocking_counts.sort_values(by=['DefensiveTeam', 'Total_Bloqueios'], ascending=[True, False])
    .groupby('DefensiveTeam')
    .first()
    .reset_index()
    .sort_values(by='Total_Bloqueios', ascending=False)
)

# Exibir a tabela com o principal bloqueador de cada time
display(top_blocker_por_time)

,DefensiveTeam,BlockingPlayer,Total_Bloqueios
30,TB,D.Watson,3
31,TEN,T.Shaw,2
3,BUF,A.Dixon,2
16,LA,C.Littleton,2
11,HOU,B.Braman,2
24,PHI,J.Casey,2
22,NYJ,E.Smith,2
17,MIA,J.Wilson,2
27,SEA,D.Baldwin,2
21,NYG,D.Moore,2


In [46]:
# Lista ordenada de todos os jogadores com bloqueios agrupados por time
jogadores_por_time = blocking_counts.sort_values(
    by=['DefensiveTeam', 'Total_Bloqueios'], 
    ascending=[True, False]
)

display(jogadores_por_time)

,DefensiveTeam,BlockingPlayer,Total_Bloqueios
0,ARI,Q.Groves,1
1,ATL,B.Williams,1
2,BAL,A.Jackson,1
3,BAL,D.Stewart,1
4,BAL,J.Allen,1
...,...,...,...
100,TB,J.George,1
103,TEN,T.Shaw,2
101,TEN,N.Schommer,1
102,TEN,P.Bailey,1


In [47]:
# 1. Filtrar registros válidos de BlockingPlayer
df_blocking = df[df['BlockingPlayer'].notna() & (df['BlockingPlayer'] != '<NA>')].copy()

# 2. Agrupar por equipa defensiva e jogador, contando os bloqueios
blocking_counts = (
    df_blocking.groupby(['DefensiveTeam', 'BlockingPlayer'])
    .size()
    .reset_index(name='Total_Bloqueios')
)

# 3. Obter o maior bloqueador de cada equipa
top_por_time = (
    blocking_counts.sort_values(by=['DefensiveTeam', 'Total_Bloqueios'], ascending=[True, False])
    .groupby('DefensiveTeam')
    .first()
    .reset_index()
)

# 4. Ordenar os times do maior para o menor número de bloqueios
times_ordenados_por_lider = top_por_time.sort_values(by='Total_Bloqueios', ascending=False).reset_index(drop=True)

display(times_ordenados_por_lider)

,DefensiveTeam,BlockingPlayer,Total_Bloqueios
0,TB,D.Watson,3
1,TEN,T.Shaw,2
2,BUF,A.Dixon,2
3,LA,C.Littleton,2
4,HOU,B.Braman,2
5,PHI,J.Casey,2
6,NYJ,E.Smith,2
7,MIA,J.Wilson,2
8,SEA,D.Baldwin,2
9,NYG,D.Moore,2


In [48]:
# 1. Agrupar diretamente por equipa e somar todos os bloqueios
total_bloqueios_time = (
    df_blocking.groupby('DefensiveTeam')
    .size()
    .reset_index(name='Total_Bloqueios_Equipe')
    .sort_values(by='Total_Bloqueios_Equipe', ascending=False)
    .reset_index(drop=True)
)

display(total_bloqueios_time)

,DefensiveTeam,Total_Bloqueios_Equipe
0,MIA,9
1,PHI,8
2,SEA,8
3,TB,8
4,OAK,7
5,SF,5
6,JAC,5
7,NYJ,4
8,DEN,4
9,STL,4


In [49]:
df.info(verbose=True)

<class 'pandas.DataFrame'>
RangeIndex: 407688 entries, 0 to 407687
Data columns (total 102 columns):
 #    Column                       Dtype   
---   ------                       -----   
 0    Date                         str     
 1    GameID                       int64   
 2    Drive                        int64   
 3    qtr                          int64   
 4    down                         float64 
 5    time                         str     
 6    TimeUnder                    int64   
 7    TimeSecs                     float64 
 8    PlayTimeDiff                 float64 
 9    SideofField                  str     
 10   yrdln                        float64 
 11   yrdline100                   float64 
 12   ydstogo                      int64   
 13   ydsnet                       int64   
 14   GoalToGo                     float64 
 15   FirstDown                    float64 
 16   posteam                      str     
 17   DefensiveTeam                str     
 18   desc         

--------------
-------------